# EDA Summary Excel Generator
Generates a comprehensive EDA Excel workbook for the AML pipeline output (`stg_transactions_features.parquet`).

**Sheets produced:**
1. Overview
2. Data Dictionary
3. Feature Formulas
4. Descriptive Stats
5. Missing Values
6. Graph Features EDA
7. Categorical Distributions
8. Rule Features
9. Temporal Analysis
10. Fraud Type Analysis
11. Top Correlations


## 1. Imports & Setup


In [1]:
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side, numbers
)
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows
from pathlib import Path
import warnings, os
warnings.filterwarnings('ignore')
print('Imports successful.')



Imports successful.


## 2. Configure Paths


In [2]:
BASE_DIR     = Path('.').resolve().parent
PARQUET_PATH = BASE_DIR / 'outputs_updated' / 'stg_transactions_features_V2.parquet'
OUTPUT_PATH  = Path('eda_output') / 'AML_EDA_Summary.xlsx'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Fallback paths
if not PARQUET_PATH.exists():
    for alt in ['stg_transactions_features_V2.parquet',
                '../outputs_updated/stg_transactions_rules_V3.parquet',
                'stg_transactions_rules_V3.parquet']:
        if Path(alt).exists():
            PARQUET_PATH = Path(alt)
            break

print(f'Input:  {PARQUET_PATH}')
print(f'Output: {OUTPUT_PATH}')



Input:  C:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated\stg_transactions_features_V2.parquet
Output: eda_output\AML_EDA_Summary.xlsx


## 3. Colour Palette


In [3]:
C_DARK_BLUE  = '1F3864'
C_MED_BLUE   = '2E75B6'
C_LIGHT_BLUE = 'D6E4F0'
C_ORANGE     = 'C55A11'
C_YELLOW     = 'FFD966'
C_GREEN      = '70AD47'
C_LIGHT_GREY = 'F2F2F2'
C_RED_LIGHT  = 'FCE4D6'
C_WHITE      = 'FFFFFF'
C_TEAL       = '00B0F0'
print('Colour constants defined.')



Colour constants defined.


## 4. Style Helper Functions


In [4]:
def hdr_font(bold=True, size=11, color=C_WHITE):
    return Font(bold=bold, size=size, color=color, name='Calibri')

def body_font(bold=False, size=10, color='000000'):
    return Font(bold=bold, size=size, color=color, name='Calibri')

def fill(hex_color):
    return PatternFill('solid', fgColor=hex_color)

def center():
    return Alignment(horizontal='center', vertical='center', wrap_text=True)

def left():
    return Alignment(horizontal='left', vertical='center', wrap_text=True)

def thin_border():
    s = Side(style='thin', color='B8B8B8')
    return Border(left=s, right=s, top=s, bottom=s)

def apply_header(ws, row_num, values, col_start=1, bg=C_DARK_BLUE, fg=C_WHITE, size=11):
    for i, val in enumerate(values, start=col_start):
        c = ws.cell(row=row_num, column=i, value=val)
        c.font = hdr_font(size=size, color=fg)
        c.fill = fill(bg)
        c.alignment = center()
        c.border = thin_border()

def apply_subheader(ws, row_num, values, col_start=1, bg=C_MED_BLUE):
    for i, val in enumerate(values, start=col_start):
        c = ws.cell(row=row_num, column=i, value=val)
        c.font = hdr_font(size=10, color=C_WHITE)
        c.fill = fill(bg)
        c.alignment = center()
        c.border = thin_border()

def write_df(ws, df, start_row, start_col=1, alt_colors=True):
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=False)):
        excel_row = start_row + r_idx
        bg = C_LIGHT_BLUE if (alt_colors and r_idx % 2 == 0) else C_WHITE
        for c_idx, val in enumerate(row, start=start_col):
            c = ws.cell(row=excel_row, column=c_idx, value=val)
            c.font = body_font()
            c.fill = fill(bg)
            c.alignment = left()
            c.border = thin_border()
    return start_row + len(df)

def set_col_widths(ws, widths):
    for col, w in widths:
        if isinstance(col, int):
            col = get_column_letter(col)
        ws.column_dimensions[col].width = w

def title_cell(ws, row, col, text, size=14, color=C_DARK_BLUE):
    c = ws.cell(row=row, column=col, value=text)
    c.font = Font(bold=True, size=size, color=color, name='Calibri')
    c.alignment = left()

def section_cell(ws, row, col, text, bg=C_MED_BLUE):
    c = ws.cell(row=row, column=col, value=text)
    c.font = Font(bold=True, size=10, color=C_WHITE, name='Calibri')
    c.fill = fill(bg)
    c.alignment = left()
    c.border = thin_border()

def write_row_styled(ws, row, values, bg=C_WHITE, bold_first=True):
    for ci, val in enumerate(values, 1):
        c = ws.cell(row=row, column=ci, value=val)
        c.font = body_font(bold=(bold_first and ci == 1))
        c.fill = fill(bg)
        c.border = thin_border()
        c.alignment = left()

print('Style helpers defined.')



Style helpers defined.


## 5. Load Data


In [5]:
print('Loading parquet ...')
df = pd.read_parquet(PARQUET_PATH)
print(f'  Shape: {df.shape}')

# Resolve key columns
COL_AMT   = next((c for c in ["transaction_amount","amount"] if c in df.columns), None)
COL_TYPE  = next((c for c in ["transaction_type_dr_cr","txn_type"] if c in df.columns), None)
COL_CHAN  = next((c for c in ["transaction_mode_channel_bank","channel_bank"] if c in df.columns), None)
COL_DS    = next((c for c in ["datestamp","Datestamp"] if c in df.columns), None)
COL_TS    = next((c for c in ["timestamp","Timestamp"] if c in df.columns), None)
COL_ACCT  = next((c for c in ["customer_account_number","account_number"] if c in df.columns), None)
COL_CIF   = next((c for c in ["customer_cif_id","cif_id"] if c in df.columns), None)
COL_CP    = next((c for c in ["counterparty_account_number","cp_account_number"] if c in df.columns), None)
COL_CASH  = next((c for c in ["cash_flag"] if c in df.columns), None)
COL_AML   = "is_aml" if "is_aml" in df.columns else None
COL_TYP   = "aml_typology" if "aml_typology" in df.columns else None
COL_RISK  = next((c for c in ["customer_current_risk_score","risk_score"] if c in df.columns), None)
COL_OCCUP = next((c for c in ["customer_occupation_industry","occupation"] if c in df.columns), None)

# Parse helper columns
if COL_DS and COL_TS:
    df["_dt"] = pd.to_datetime(df[COL_DS].astype(str) + " " + df[COL_TS].astype(str),
                               format="%d-%m-%Y %H:%M:%S", errors="coerce")
    df["txn_hour"] = df["_dt"].dt.hour
    df["txn_day_of_week"] = df["_dt"].dt.dayofweek
    df["txn_month"] = df["_dt"].dt.month

if COL_AMT:
    df["_amt"] = pd.to_numeric(df[COL_AMT], errors="coerce").fillna(0)

print(f'  Columns resolved. AML column: {COL_AML}')
df.head(3)



Loading parquet ...
  Shape: (386570, 318)
  Columns resolved. AML column: is_aml


,transaction_id,timestamp,datestamp,transaction_amount,currency,transaction_type_dr_cr,transaction_mode_channel_bank,cash_flag,transaction_type_ppi,transaction_mode_channel_ppi,...,convergence_risk,temporal_risk,fraud_intensity_score_raw,fraud_intensity_score,fis_band,_dt,txn_hour,txn_day_of_week,txn_month,_amt
0,TXNFV7CXKNM3PDK3CS1,08:52:21,01-12-2025,4352.33,INR,Cr,UPI,N,,,...,0.0000,0.0,0.3750,0.38,very_low,2025-12-01 08:52:21,8,0,12,4352.33
1,TXN10WAJMRY09HZS1UF,22:39:50,03-12-2025,519.74,INR,Dr,Internet Banking,N,,,...,0.0000,0.2,1.8750,1.88,very_low,2025-12-03 22:39:50,22,2,12,519.74
2,TXNCAK4V1YXDA1Q9YW0,17:58:22,04-12-2025,1839.19,INR,Cr,Demand Draft,N,,,...,0.1932,0.0,8.4047,8.40,very_low,2025-12-04 17:58:22,17,3,12,1839.19


## 6. Data Dictionary Definition


In [6]:
DATA_DICT = {
    # Transaction Core
    "transaction_id":           ("string",  "Transaction Base", "Unique transaction reference number", "TXN + 16 alphanumeric chars"),
    "timestamp":                ("string",  "Transaction Base", "Transaction time HH:MM:SS", "N/A"),
    "datestamp":                ("string",  "Transaction Base", "Transaction date DD-MM-YYYY", "N/A"),
    "transaction_amount":       ("float",   "Transaction Base", "Monetary value of transaction in INR", "N/A"),
    "currency":                 ("string",  "Transaction Base", "ISO 4217 currency code", "INR (97%), FX (3%)"),
    "transaction_type_dr_cr":   ("string",  "Transaction Base", "Debit (Dr) or Credit (Cr)", "Dr 55%, Cr 45%"),
    "transaction_mode_channel_bank": ("string", "Transaction Base", "Bank channel: NEFT, RTGS, IMPS, UPI, etc.", "N/A"),
    "cash_flag":                ("string",  "Transaction Base", "Y if cash transaction, N otherwise", "Y if Branch Cash or ATM"),
    "transaction_status":       ("string",  "Transaction Base", "Success, Failed, Pending, Reversed", "Success 92%"),
    # Participant
    "customer_account_number":  ("string",  "Participant", "Customer bank account number", "N/A"),
    "customer_cif_id":          ("string",  "Participant", "Customer Information File identifier", "N/A"),
    "counterparty_account_number": ("string","Participant", "Beneficiary/sender account", "N/A"),
    "sender_country_code":      ("string",  "Participant", "Sender ISO country code", "IN or foreign"),
    "receiver_country_code":    ("string",  "Participant", "Receiver ISO country code", "IN or foreign"),
    "customer_current_risk_score": ("string","Customer Profile", "Dynamic risk rating: Low/Medium/High", "Low 60%, Med 30%, High 10%"),
    "customer_type":            ("string",  "Customer Profile", "Individual or Non-Individual", "Individual 80%"),
    "customer_occupation_industry": ("string","Customer Profile", "Occupation or industry sector", "N/A"),
    "pep_flag":                 ("string",  "Customer Profile", "Politically Exposed Person flag", "Y 2%"),
    "vkyc_flag":                ("string",  "Customer Profile", "Video KYC completed", "Y 25%"),
    "annual_income":            ("integer", "Customer Profile", "Self-declared annual income INR", "N/A"),
    "vpn_flag":                 ("string",  "Device", "VPN/proxy detected", "Y 3%"),
    "emulator_flag":            ("string",  "Device", "Emulator detected", "Y 1%"),
    "device_id_fingerprint":    ("string",  "Device", "Device fingerprint hash", "N/A"),
    "ip_address":               ("string",  "Device", "Originating IP address", "N/A"),
    # Labels
    "is_aml":                   ("integer", "Labels", "AML flag: 0=clean, 1=flagged", "Set by typology detector"),
    "aml_typology":             ("string",  "Labels", "Detected typology name(s)", "Empty if is_aml=0"),
    "typology_group_id":        ("string",  "Labels", "Scenario group identifier", "PREFIX_00001 format"),
    # Rule aggregates
    "rule_score":               ("integer", "Rule Aggregates", "Composite risk score", "SUM(rule_flag * severity)"),
    "rules_triggered":          ("string",  "Rule Aggregates", "Semicolon-separated triggered rule names", "N/A"),
    "rules_triggered_count":    ("integer", "Rule Aggregates", "Count of rules triggered", "0 to 50"),
    "alert_level":              ("string",  "Rule Aggregates", "Alert classification", "Critical>=9, High>=6, Medium>=3, Low>=1, None=0"),
    # Velocity
    "sender_acct_txn_count_1h": ("integer", "Account Velocity", "Txn count from account in last 1h", "Rolling 1h window"),
    "sender_acct_txn_count_24h":("integer", "Account Velocity", "Txn count from account in last 24h", "Rolling 24h window"),
    "sender_acct_outflow_amt_24h":("float", "Account Velocity", "Total debit amount in last 24h", "Rolling 24h window"),
    "sender_acct_inflow_amt_24h": ("float", "Account Velocity", "Total credit amount in last 24h", "Rolling 24h window"),
    "sender_cust_txn_count_24h":("integer", "Customer Velocity", "Customer txn count (all accounts) in 24h", "Rolling 24h window"),
    # Balance
    "sender_balance_before_txn":("float",   "Balance", "Running balance before this transaction", "Cumulative from first txn"),
    "sender_balance_after_txn": ("float",   "Balance", "Running balance after this transaction", "before + signed_amount"),
    "sender_current_balance":   ("float",   "Balance", "Latest known account balance", "= last balance_after_txn"),
    # Receiver
    "receiver_acct_txn_count_24h":("integer","Receiver", "Counterparty txn count in 24h", "Mapped from counterparty's sender metrics"),
    "receiver_account_outflow_30d":("integer","Receiver","Receiver outflow count 30d", "= receiver_acct_outflow_count_30d"),
    # Ratios
    "inflow_outflow_volume_balance_ratio_24h":("float","Graph Ratios","24h volume balance ratio","min(inflow,outflow)/inflow; near 1=pass-through"),
    "inflow_outflow_volume_balance_ratio_7d": ("float","Graph Ratios","7d volume balance ratio","Same over 7-day window"),
    # IP & FIS
    "ip_risk_score":            ("float",   "IP Risk", "Composite IP risk 0.0-1.0", "base(0.10)+vpn(0.15)+emulator(0.10)+night(0.10)+country(0.10)+cross_border(0.05)+shared_ip(0.10)+geo_mismatch(0.10)-kyc(0.10)"),
    "ip_flag_vpn":              ("integer", "IP Risk", "VPN detected component", "+0.15 to ip_risk_score"),
    "ip_flag_emulator":         ("integer", "IP Risk", "Emulator detected component", "+0.10"),
    "ip_flag_night":            ("integer", "IP Risk", "Night transaction component", "+0.10"),
    "ip_flag_country_high_risk":("integer", "IP Risk", "FATF country component", "+0.10"),
    "ip_flag_cross_border":     ("integer", "IP Risk", "Cross-border component", "+0.05"),
    "ip_flag_shared_ip":        ("integer", "IP Risk", "Shared IP component", "+0.10 scaled"),
    "ip_flag_geo_mismatch":     ("integer", "IP Risk", "Geo mismatch component", "+0.10"),
    "ip_flag_kyc_verified":     ("integer", "IP Risk", "KYC verified (reduces risk)", "-0.10"),
    "fraud_intensity_score_raw":("float",   "FIS", "Raw FIS before normalization", "rule*35+behaviour*30+ip*18+device*17"),
    "fraud_intensity_score":    ("float",   "FIS", "Normalized FIS 0-100", "MIN(raw/p99, 1)*100"),
    "fis_band":                 ("string",  "FIS", "FIS band", "very_low/low/medium/high/critical"),
}

# Auto-add all rule_ columns not already in dict
for c in df.columns:
    if c.startswith("rule_") and c not in DATA_DICT:
        DATA_DICT[c] = ("integer", "Rule Flags", c.replace("rule_","").replace("_"," ").title(), "0 or 1")

# Auto-add velocity/balance columns not already in dict
for c in df.columns:
    if c not in DATA_DICT:
        if "sender_acct" in c:
            DATA_DICT[c] = ("float/int", "Account Velocity", c.replace("_"," ").title(), "Rolling window")
        elif "sender_cust" in c:
            DATA_DICT[c] = ("float/int", "Customer Velocity", c.replace("_"," ").title(), "Rolling window")
        elif "receiver_" in c:
            DATA_DICT[c] = ("float/int", "Receiver", c.replace("_"," ").title(), "From counterparty")
        elif "sender_bal" in c or "sender_balance" in c or "sender_current" in c or "sender_running" in c or "sender_cumulative" in c:
            DATA_DICT[c] = ("float", "Balance", c.replace("_"," ").title(), "Cumulative tracking")

print(f'Data dictionary: {len(DATA_DICT)} entries')



Data dictionary: 245 entries


## 7. Feature Formulas Definition


In [7]:
FORMULAS = [
    ("Amount Derived", "transaction_amount", "Raw amount in INR", "Core feature for all rules and scoring", "All typologies"),
    ("Temporal", "txn_hour", "timestamp.hour", "Night hours (22-06) indicate reduced oversight", "Layering, ATO, Structuring"),
    ("Temporal", "txn_day_of_week", "timestamp.dayofweek (0=Mon)", "Weekend transactions receive less scrutiny", "All typologies"),
    ("Account Velocity", "sender_acct_txn_count_{1h,24h,7d,30d}", "Rolling count per account within window", "Rapid bursts indicate automated/scripted activity", "Mule Ring, Smurfing, Layering"),
    ("Account Velocity", "sender_acct_inflow_amt_{1h,24h,7d,30d}", "Rolling SUM(Cr amounts) per account", "Large inflows before rapid outflows = pass-through", "Pass-Through, Funnel"),
    ("Account Velocity", "sender_acct_outflow_amt_{1h,24h,7d,30d}", "Rolling SUM(Dr amounts) per account", "Large outflows signal fund movement or extraction", "Layering, Mule Ring"),
    ("Customer Velocity", "sender_cust_txn_count_{1h,24h,7d,30d}", "Rolling count per CIF across all accounts", "Customer-level velocity catches multi-account schemes", "Structuring, Mule Ring"),
    ("Customer Velocity", "sender_cust_outflow_amt_{1h,24h,7d,30d}", "Rolling SUM(Dr amounts) per CIF", "Customer draining funds across accounts", "Layering, Funnel"),
    ("Balance", "sender_balance_before_txn", "Cumulative running balance from first txn", "Starting balance context for ratio features", "All"),
    ("Balance", "sender_balance_after_txn", "before + signed_amount (+Cr/-Dr)", "Post-txn balance; negative = account drained", "ATO, Mule Ring"),
    ("Balance", "sender_bal_ratio_after_to_current", "balance_after / current_balance", "Values < 0 indicate drain below zero", "ATO, Layering"),
    ("Graph Ratio", "inflow_outflow_volume_balance_ratio_24h", "MIN(inflow_24h, outflow_24h) / inflow_24h", "Near 1.0 = all received funds immediately forwarded", "Pass-Through, Layering"),
    ("Graph Ratio", "inflow_outflow_volume_balance_ratio_7d", "Same over 7-day window", "Captures slower layering patterns", "Layering, Funnel"),
    ("IP Risk", "ip_risk_score", "0.10 + vpn(0.15) + emulator(0.10) + night(0.10) + country(0.10) + cross_border(0.05) + shared_ip(0.10) + geo_mismatch(0.10) - kyc(0.10), clipped [0,1]", "Composite signal from network/device/location anomalies", "All typologies"),
    ("FIS", "fraud_intensity_score", "MIN(raw_score / p99, 1) * 100", "Normalized composite: rule*35 + behaviour*30 + ip*18 + device*17", "All typologies"),
    ("FIS", "fis_band", "0-20=very_low, 20-40=low, 40-60=medium, 60-80=high, 80-100=critical", "Discretized band for operational routing", "All typologies"),
]
print(f'Feature formulas: {len(FORMULAS)}')



Feature formulas: 16


## 8. Sheet 1 -- Overview


In [8]:
def sheet_overview(wb, df):
    ws = wb.create_sheet("1_Overview")
    ws.freeze_panes = "B3"
    title_cell(ws, 1, 1, "AML Dataset - EDA Overview", size=16)

    amt = df["_amt"] if "_amt" in df.columns else pd.Series(dtype=float)
    aml_count = int((df[COL_AML] == 1).sum()) if COL_AML else 0
    clean_count = len(df) - aml_count

    metrics = [
        ("Total Transactions",       f"{len(df):,}"),
        ("Total Columns / Features", f"{df.shape[1]}"),
        ("Date Range Start",         str(df[COL_DS].min()) if COL_DS else "N/A"),
        ("Date Range End",           str(df[COL_DS].max()) if COL_DS else "N/A"),
        ("AML Transactions (is_aml=1)", f"{aml_count:,}"),
        ("Clean (is_aml=0)",         f"{clean_count:,}"),
        ("Overall AML Rate",         f"{aml_count/len(df)*100:.2f}%"),
        ("Unique Accounts",          f"{df[COL_ACCT].nunique():,}" if COL_ACCT else "N/A"),
        ("Unique Customers (CIF)",   f"{df[COL_CIF].nunique():,}" if COL_CIF else "N/A"),
        ("Unique Channels",          f"{df[COL_CHAN].nunique()}" if COL_CHAN else "N/A"),
        ("Columns with Nulls",       f"{(df.isnull().sum()>0).sum()}"),
        ("Total Missing Cells",      f"{df.isnull().sum().sum():,}"),
        ("Mean Amount (INR)",        f"{amt.mean():,.0f}" if len(amt) > 0 else "N/A"),
        ("Median Amount (INR)",      f"{amt.median():,.0f}" if len(amt) > 0 else "N/A"),
    ]

    apply_header(ws, 3, ["Metric", "Value"], bg=C_DARK_BLUE)
    for i, (k, v) in enumerate(metrics, start=4):
        bg = C_LIGHT_BLUE if i % 2 == 0 else C_WHITE
        write_row_styled(ws, i, [k, v], bg=bg)

    # Typology distribution
    row = 4 + len(metrics) + 2
    if COL_TYP and COL_AML:
        title_cell(ws, row, 1, "Typology Distribution", size=12)
        row += 1
        apply_header(ws, row, ["Typology", "Txn Count", "% of All Txns", "% of AML Txns"], bg=C_MED_BLUE)
        row += 1
        aml_df = df[df[COL_AML] == 1]
        from collections import Counter
        all_typs = []
        for t in aml_df[COL_TYP].dropna():
            for part in str(t).split("; "):
                if part.strip():
                    all_typs.append(part.strip())
        tc = Counter(all_typs)
        for typ, cnt in tc.most_common():
            bg = C_RED_LIGHT if cnt > 1000 else C_WHITE
            write_row_styled(ws, row, [typ, cnt, f"{cnt/len(df)*100:.2f}%", f"{cnt/max(len(aml_df),1)*100:.1f}%"], bg=bg)
            row += 1

    # Channel distribution
    row += 2
    if COL_CHAN and COL_AML:
        title_cell(ws, row, 1, "Channel Distribution", size=12)
        row += 1
        apply_header(ws, row, ["Channel", "Total", "AML Count", "AML Rate (%)"], bg=C_MED_BLUE)
        row += 1
        ch = df.groupby(COL_CHAN)[COL_AML].agg(["count","sum","mean"]).reset_index()
        ch.columns = ["channel","total","aml_count","aml_rate"]
        ch = ch.sort_values("total", ascending=False)
        for _, r in ch.iterrows():
            bg = C_LIGHT_BLUE if row % 2 == 0 else C_WHITE
            write_row_styled(ws, row, [r["channel"], int(r["total"]), int(r["aml_count"]), f"{r['aml_rate']*100:.2f}%"], bg=bg)
            row += 1

    set_col_widths(ws, [(1, 40), (2, 25), (3, 20), (4, 20)])
    ws.sheet_view.showGridLines = False
    print("  ✓ Sheet 1_Overview")



## 9. Sheet 2 -- Data Dictionary


In [9]:
def sheet_data_dict(wb, df):
    ws = wb.create_sheet("2_Data_Dictionary")
    ws.freeze_panes = "A3"
    title_cell(ws, 1, 1, "Data Dictionary - All Features & Definitions")

    headers = ["Column Name","Data Type","Feature Group","Definition","Formula / Derivation"]
    apply_header(ws, 2, headers, bg=C_DARK_BLUE)

    group_colors = {
        "Transaction Base":"E8F5E9", "Labels":"FCE4D6",
        "Participant":"E3F2FD", "Customer Profile":"E3F2FD",
        "Device":"FFF9C4", "Rule Flags":"FFF8E1", "Rule Aggregates":"FFF8E1",
        "Account Velocity":"E8EAF6", "Customer Velocity":"E8EAF6",
        "Balance":"E0F7FA", "Receiver":"E8F5E9",
        "Graph Ratios":"FCE4D6", "IP Risk":"F3E5F5", "FIS":"F3E5F5",
    }

    row = 3
    for col_name in df.columns:
        if col_name.startswith("_"):
            continue
        if col_name in DATA_DICT:
            dtype, group, defn, formula = DATA_DICT[col_name]
        else:
            dtype = str(df[col_name].dtype)
            group = "Other"
            defn = ""
            formula = ""
        bg = group_colors.get(group, "FFFFFF")
        vals = [col_name, dtype, group, defn, formula]
        for ci, val in enumerate(vals, 1):
            c = ws.cell(row=row, column=ci, value=val)
            c.font = body_font(bold=(ci == 1))
            c.fill = fill(bg)
            c.border = thin_border()
            c.alignment = Alignment(horizontal="left", vertical="top", wrap_text=True)
        row += 1

    set_col_widths(ws, [(1,35),(2,12),(3,22),(4,55),(5,65)])
    ws.sheet_view.showGridLines = False
    print("  ✓ Sheet 2_Data_Dictionary")



## 10. Sheet 3 -- Feature Formulas


In [10]:
def sheet_formulas(wb):
    ws = wb.create_sheet("3_Feature_Formulas")
    ws.freeze_panes = "A3"
    title_cell(ws, 1, 1, "Feature Formulas, Derivations & AML Intuition")

    headers = ["Feature Group","Feature Name","Mathematical Formula","AML Intuition","Typologies Targeted"]
    apply_header(ws, 2, headers, bg=C_DARK_BLUE)

    group_colors = {
        "Amount Derived":"FFF3E0", "Temporal":"E0F7FA",
        "Account Velocity":"E8EAF6", "Customer Velocity":"E8EAF6",
        "Balance":"E0F7FA", "Graph Ratio":"FCE4D6",
        "IP Risk":"F3E5F5", "FIS":"F3E5F5",
    }

    row = 3
    for group, feat, formula, intuition, target in FORMULAS:
        bg = group_colors.get(group, C_WHITE)
        vals = [group, feat, formula, intuition, target]
        for ci, val in enumerate(vals, 1):
            c = ws.cell(row=row, column=ci, value=val)
            c.font = body_font(bold=(ci <= 2))
            c.fill = fill(bg)
            c.border = thin_border()
            c.alignment = Alignment(horizontal="left", vertical="top", wrap_text=True)
        ws.row_dimensions[row].height = 50
        row += 1

    set_col_widths(ws, [(1,22),(2,42),(3,60),(4,65),(5,30)])
    ws.sheet_view.showGridLines = False
    print("  ✓ Sheet 3_Feature_Formulas")



## 11. Sheet 4 -- Descriptive Stats


In [11]:
def sheet_desc_stats(wb, df):
    ws = wb.create_sheet("4_Descriptive_Stats")
    ws.freeze_panes = "A3"
    title_cell(ws, 1, 1, "Descriptive Statistics - All Numeric Features")

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if not c.startswith("_")]

    stats = df[numeric_cols].describe(percentiles=[.25,.5,.75,.90,.95,.99]).T.round(4)
    stats.insert(0, "feature", stats.index)
    stats = stats.reset_index(drop=True)
    stats["null_count"] = df[numeric_cols].isnull().sum().values
    stats["null_pct"] = (df[numeric_cols].isnull().mean() * 100).round(2).values

    headers = list(stats.columns)
    apply_header(ws, 2, headers, bg=C_DARK_BLUE)

    for r_i, row_data in enumerate(stats.itertuples(index=False), start=3):
        bg = C_LIGHT_BLUE if r_i % 2 == 0 else C_WHITE
        for c_i, val in enumerate(row_data, 1):
            c = ws.cell(row=r_i, column=c_i, value=val)
            c.font = body_font(bold=(c_i == 1))
            c.fill = fill(bg)
            c.border = thin_border()
            c.alignment = left()

    for ci in range(1, len(headers)+1):
        ws.column_dimensions[get_column_letter(ci)].width = 14
    ws.column_dimensions["A"].width = 42
    ws.sheet_view.showGridLines = False
    print("  ✓ Sheet 4_Descriptive_Stats")



## 12. Sheet 5 -- Missing Values


In [12]:
def sheet_missing(wb, df):
    ws = wb.create_sheet("5_Missing_Values")
    ws.freeze_panes = "A3"
    title_cell(ws, 1, 1, "Missing Value Analysis")

    headers = ["Column","Data Type","Null Count","Null %","Feature Group","Notes"]
    apply_header(ws, 2, headers, bg=C_DARK_BLUE)

    nulls = df.isnull().sum().reset_index()
    nulls.columns = ["column","null_count"]
    nulls["null_pct"] = (nulls["null_count"] / len(df) * 100).round(2)
    nulls["dtype"] = nulls["column"].map(lambda c: str(df[c].dtype))
    nulls["group"] = nulls["column"].map(lambda c: DATA_DICT.get(c, ("","","",""))[1])
    nulls = nulls.sort_values("null_count", ascending=False)

    row = 3
    for _, r in nulls.iterrows():
        if r["column"].startswith("_"):
            continue
        pct = r["null_pct"]
        if pct > 50:   bg = "FCE4D6"
        elif pct > 5:  bg = "FFF9C4"
        elif pct > 0:  bg = "FFFDE7"
        else:          bg = C_WHITE
        vals = [r["column"], r["dtype"], r["null_count"], f"{pct}%", r["group"], ""]
        write_row_styled(ws, row, vals, bg=bg, bold_first=False)
        row += 1

    set_col_widths(ws, [(1,38),(2,14),(3,14),(4,12),(5,22),(6,50)])
    ws.sheet_view.showGridLines = False
    print("  ✓ Sheet 5_Missing_Values")



## 13. Sheet 6 -- Graph Features EDA


In [13]:
def sheet_graph_eda(wb, df):
    ws = wb.create_sheet("6_Graph_Features_EDA")
    ws.freeze_panes = "A4"
    title_cell(ws, 1, 1, "Graph & Velocity Features - EDA & AML Signal Analysis")

    graph_cols = [c for c in df.columns if any(c.startswith(p) for p in
                  ["sender_acct_","sender_cust_","sender_bal","sender_balance","sender_current",
                   "receiver_acct","receiver_balance","receiver_current",
                   "inflow_outflow_volume","ip_risk_score","fraud_intensity_score"])
                  and not c.startswith("_")]
    graph_cols = [c for c in graph_cols if c in df.columns and df[c].dtype in ['float64','int64','float32','int32']]

    # Section 1: Descriptive stats
    section_cell(ws, 3, 1, "Section 1 - Descriptive Statistics", bg=C_MED_BLUE)
    hdrs = ["Feature","Non-Null","Mean","Std Dev","Min","25%","Median","75%","90%","99%","Max","Null %"]
    apply_header(ws, 4, hdrs, bg=C_DARK_BLUE)
    row = 5
    for col in graph_cols[:40]:
        s = df[col].dropna()
        if len(s) == 0: continue
        null_pct = df[col].isnull().mean()*100
        vals = [col, len(s), round(s.mean(),4), round(s.std(),4),
                round(s.min(),4), round(s.quantile(.25),4), round(s.median(),4),
                round(s.quantile(.75),4), round(s.quantile(.90),4), round(s.quantile(.99),4),
                round(s.max(),4), f"{null_pct:.1f}%"]
        bg = C_LIGHT_BLUE if row%2==0 else C_WHITE
        write_row_styled(ws, row, vals, bg=bg)
        row += 1

    # Section 2: AML vs Clean comparison
    if COL_AML:
        row += 2
        section_cell(ws, row, 1, "Section 2 - AML vs Clean Mean Comparison", bg=C_ORANGE)
        row += 1
        hdrs2 = ["Feature","Clean Mean","AML Mean","AML/Clean Ratio","Signal Direction"]
        apply_header(ws, row, hdrs2, bg=C_DARK_BLUE)
        row += 1
        aml_mask = df[COL_AML] == 1
        for col in graph_cols[:40]:
            clean_mean = df.loc[~aml_mask, col].mean()
            aml_mean = df.loc[aml_mask, col].mean()
            ratio = aml_mean / clean_mean if clean_mean != 0 else 0
            direction = "↑ AML higher" if ratio > 1.1 else ("↓ AML lower" if ratio < 0.9 else "≈ Similar")
            bg = C_RED_LIGHT if ratio > 2 else (C_YELLOW if ratio > 1.3 else C_WHITE)
            write_row_styled(ws, row, [col, round(clean_mean,4), round(aml_mean,4),
                             f"{ratio:.2f}x", direction], bg=bg)
            row += 1

    for ci in range(1, 13):
        ws.column_dimensions[get_column_letter(ci)].width = 16
    ws.column_dimensions["A"].width = 42
    ws.sheet_view.showGridLines = False
    print("  ✓ Sheet 6_Graph_Features_EDA")



## 14. Sheet 7 -- Categorical Distributions


In [14]:
def sheet_categoricals(wb, df):
    ws = wb.create_sheet("7_Categorical_Distributions")
    title_cell(ws, 1, 1, "Categorical Feature Distributions & AML Rates")

    cat_cols = [(COL_CHAN, "Channel"), (COL_TYPE, "Transaction Type"),
                (COL_RISK, "Customer Risk"), (COL_OCCUP, "Occupation"),
                (COL_CASH, "Cash Flag"),
                ("alert_level", "Alert Level"), ("fis_band", "FIS Band")]

    row = 3
    for col, label in cat_cols:
        if not col or col not in df.columns:
            continue
        section_cell(ws, row, 1, f"■ {label} ({col})", bg=C_MED_BLUE)
        for ci in range(2, 7):
            ws.cell(row=row, column=ci).fill = fill(C_MED_BLUE)
            ws.cell(row=row, column=ci).border = thin_border()
        row += 1
        apply_header(ws, row, ["Value","Count","% of Total","AML Count","AML Rate (%)"], bg=C_DARK_BLUE)
        row += 1

        vc = df[col].astype(str).value_counts().head(20)
        for val, cnt in vc.items():
            aml_cnt = int((df[df[col].astype(str) == val][COL_AML] == 1).sum()) if COL_AML else 0
            aml_rate = aml_cnt / cnt * 100 if cnt > 0 else 0
            bg = C_RED_LIGHT if aml_rate > 20 else (C_LIGHT_BLUE if row % 2 == 0 else C_WHITE)
            write_row_styled(ws, row, [val, cnt, f"{cnt/len(df)*100:.2f}%", aml_cnt, f"{aml_rate:.2f}%"], bg=bg)
            row += 1
        row += 1

    set_col_widths(ws, [(1,30),(2,12),(3,14),(4,12),(5,14)])
    ws.sheet_view.showGridLines = False
    print("  ✓ Sheet 7_Categorical_Distributions")



## 15. Sheet 8 -- Rule Features


In [15]:
def sheet_rules(wb, df):
    ws = wb.create_sheet("8_Rule_Features")
    ws.freeze_panes = "A4"
    title_cell(ws, 1, 1, "Rule-Based Feature Analysis - Trigger Rates & AML Correlation")

    rule_cols = [c for c in df.columns if c.startswith("rule_") and c not in
                 ["rule_score","rules_triggered","rules_triggered_count"]]

    section_cell(ws, 3, 1, "Section 1 - Per-Rule Statistics", bg=C_MED_BLUE)
    hdrs = ["Rule Name","Trigger Count","Trigger Rate (%)","AML Rate When Triggered (%)","AML Rate When NOT Triggered (%)","Lift","Signal Strength"]
    apply_header(ws, 4, hdrs, bg=C_DARK_BLUE)
    row = 5

    rule_stats = []
    for c in rule_cols:
        total = int(df[c].astype(float).sum())
        tr = df[c].astype(float).mean() * 100
        ft = df.loc[df[c].astype(float)==1, COL_AML].mean() * 100 if total > 0 and COL_AML else 0
        fn = df.loc[df[c].astype(float)==0, COL_AML].mean() * 100 if COL_AML else 0
        lift = ft / fn if fn > 0 else 0
        rule_stats.append((c, total, tr, ft, fn, lift))

    rule_stats.sort(key=lambda x: x[3], reverse=True)

    for (c, cnt, tr, ft, fn, lift) in rule_stats:
        if ft > 50:    bg, sig = "FCE4D6", "Very High"
        elif ft > 30:  bg, sig = "FFF3E0", "High"
        elif ft > 15:  bg, sig = "FFF9C4", "Medium"
        else:          bg, sig = C_WHITE,  "Low"
        write_row_styled(ws, row, [c, cnt, f"{tr:.2f}%", f"{ft:.1f}%", f"{fn:.1f}%",
                         f"{lift:.2f}x" if lift > 0 else "N/A", sig], bg=bg)
        row += 1

    # Section 2: Alert level summary
    row += 2
    section_cell(ws, row, 1, "Section 2 - Alert Level Distribution", bg=C_MED_BLUE)
    row += 1
    if "alert_level" in df.columns:
        apply_header(ws, row, ["Alert Level","Count","% of Total","AML Count","AML Rate (%)"], bg=C_DARK_BLUE)
        row += 1
        for level in ["Critical","High","Medium","Low","None"]:
            mask = df["alert_level"] == level
            cnt = mask.sum()
            aml_cnt = int(df.loc[mask, COL_AML].sum()) if COL_AML else 0
            bg = C_RED_LIGHT if level == "Critical" else (C_YELLOW if level == "High" else C_WHITE)
            write_row_styled(ws, row, [level, cnt, f"{cnt/len(df)*100:.1f}%", aml_cnt,
                             f"{aml_cnt/max(cnt,1)*100:.1f}%"], bg=bg)
            row += 1

    set_col_widths(ws, [(1,42),(2,14),(3,16),(4,28),(5,30),(6,10),(7,16)])
    ws.sheet_view.showGridLines = False
    print("  ✓ Sheet 8_Rule_Features")



## 16. Sheet 9 -- Temporal Analysis


In [16]:
def sheet_temporal(wb, df):
    ws = wb.create_sheet("9_Temporal_Analysis")
    title_cell(ws, 1, 1, "Temporal Analysis - AML Patterns Over Time")

    row = 3
    # Hour of day
    if "txn_hour" in df.columns and COL_AML:
        section_cell(ws, row, 1, "AML Rate by Hour of Day", bg=C_MED_BLUE)
        row += 1
        apply_header(ws, row, ["Hour","Total Txns","AML Count","AML Rate (%)","Period"], bg=C_DARK_BLUE)
        row += 1
        hour_tbl = df.groupby("txn_hour")[COL_AML].agg(["count","sum","mean"]).reset_index()
        for _, r in hour_tbl.iterrows():
            h = int(r["txn_hour"])
            period = ("Night (22-05)" if (h >= 22 or h < 6)
                      else ("Morning (6-9)" if h < 9
                      else ("Business (9-17)" if h < 17
                      else "Evening (17-21)")))
            bg = "FCE4D6" if r["mean"] > 0.15 else (C_LIGHT_BLUE if row%2==0 else C_WHITE)
            write_row_styled(ws, row, [h, int(r["count"]), int(r["sum"]),
                             f"{r['mean']*100:.1f}%", period], bg=bg)
            row += 1

    # Day of week
    row += 2
    if "txn_day_of_week" in df.columns and COL_AML:
        section_cell(ws, row, 1, "AML Rate by Day of Week", bg=C_MED_BLUE)
        row += 1
        apply_header(ws, row, ["Day (0=Mon)","Day Name","Total","AML Count","AML Rate (%)"], bg=C_DARK_BLUE)
        row += 1
        days = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
        dow = df.groupby("txn_day_of_week")[COL_AML].agg(["count","sum","mean"]).reset_index()
        for _, r in dow.iterrows():
            d = int(r["txn_day_of_week"])
            bg = C_LIGHT_BLUE if row%2==0 else C_WHITE
            write_row_styled(ws, row, [d, days[d], int(r["count"]), int(r["sum"]),
                             f"{r['mean']*100:.1f}%"], bg=bg)
            row += 1

    # Monthly
    row += 2
    if "txn_month" in df.columns and COL_AML:
        section_cell(ws, row, 1, "AML Rate by Month", bg=C_MED_BLUE)
        row += 1
        apply_header(ws, row, ["Month","Total","AML Count","AML Rate (%)"], bg=C_DARK_BLUE)
        row += 1
        monthly = df.groupby("txn_month")[COL_AML].agg(["count","sum","mean"]).reset_index()
        for _, r in monthly.iterrows():
            bg = C_LIGHT_BLUE if row%2==0 else C_WHITE
            write_row_styled(ws, row, [int(r["txn_month"]), int(r["count"]), int(r["sum"]),
                             f"{r['mean']*100:.1f}%"], bg=bg)
            row += 1

    set_col_widths(ws, [(1,14),(2,16),(3,14),(4,14),(5,18)])
    ws.sheet_view.showGridLines = False
    print("  ✓ Sheet 9_Temporal_Analysis")



## 17. Sheet 10 -- Fraud Type Analysis


In [17]:
def sheet_fraud_types(wb, df):
    ws = wb.create_sheet("10_Fraud_Type_Analysis")
    title_cell(ws, 1, 1, "Typology Deep-Dive Analysis")

    if not COL_TYP or not COL_AML:
        ws.cell(row=3, column=1, value="No typology column found").font = body_font()
        print("  ✓ Sheet 10_Fraud_Type_Analysis (skipped)")
        return

    rule_cols = [c for c in df.columns if c.startswith("rule_") and c not in
                 ["rule_score","rules_triggered","rules_triggered_count"]]

    typology_desc = {
        "Structuring (Smurfing)": "Cash broken into sub-threshold deposits across accounts, then consolidated to single target. Classic FATF placement typology.",
        "Circular Transaction Loop": "Funds move in a ring (A→B→C→A) with similar amounts. Simulates fake business activity for layering.",
        "Funnel Account Network": "15-50 unrelated senders funnel money to one account which rapidly forwards 95% to a destination.",
        "Pass-Through Transit Hub": "Account receives large sum and forwards 96-99% within minutes. Near-zero net position.",
        "Rapid Multi-Hop Layering": "Funds traverse 8-10 accounts within hours, each hop deducting 1-2%. Obscures provenance.",
        "Third-Party Payment Web": "Business receives payments from unrelated individuals not matching its customer base.",
        "Money Mule Network": "Controller sends to 5-20 recruited mules who forward 85-95% to a collector within hours.",
        "High-Risk Corridor Transfer": "Repeated transfers to FATF high-risk countries (AE, PK, BD, NP, LK, MM, AF).",
        "Underground Banking (Hawala)": "Triangular/quadrilateral off-books settlement with amounts within 5% tolerance.",
        "Charity Abuse": "NPO collects donations from 10-40 donors then diverts 80% to personal accounts.",
    }

    aml_df = df[df[COL_AML] == 1]
    from collections import Counter
    all_typs = []
    for t in aml_df[COL_TYP].dropna():
        for part in str(t).split("; "):
            if part.strip(): all_typs.append(part.strip())
    unique_typs = sorted(set(all_typs))

    row = 3
    for ft in unique_typs:
        sub = df[df[COL_TYP].astype(str).str.contains(ft, na=False)]
        clean = df[df[COL_AML] != 1]

        section_cell(ws, row, 1, f"■  {ft.upper()}  (n = {len(sub):,})", bg=C_DARK_BLUE)
        for ci in range(2, 7):
            ws.cell(row=row, column=ci).fill = fill(C_DARK_BLUE)
            ws.cell(row=row, column=ci).border = thin_border()
        row += 1

        desc = typology_desc.get(ft, "")
        c = ws.cell(row=row, column=1, value=desc)
        c.font = Font(italic=True, size=10, name="Calibri", color="333333")
        c.alignment = Alignment(wrap_text=True)
        ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=6)
        ws.row_dimensions[row].height = 40
        row += 1

        # Key stats
        apply_header(ws, row, ["Metric","Value"], bg=C_MED_BLUE, size=10)
        row += 1
        amt_vals = pd.to_numeric(sub[COL_AMT], errors="coerce") if COL_AMT else pd.Series()
        stats = [
            ("Transaction Count", f"{len(sub):,}"),
            ("Mean Amount", f"INR {amt_vals.mean():,.0f}" if len(amt_vals)>0 else "N/A"),
            ("Median Amount", f"INR {amt_vals.median():,.0f}" if len(amt_vals)>0 else "N/A"),
            ("Total Amount", f"INR {amt_vals.sum():,.0f}" if len(amt_vals)>0 else "N/A"),
        ]
        for k, v in stats:
            write_row_styled(ws, row, [k, v], bg=C_LIGHT_BLUE)
            row += 1

        # Top triggered rules
        row += 1
        apply_header(ws, row, ["Top Rules","Trigger Rate (%)","vs Clean Rate (%)","Lift"], bg=C_MED_BLUE, size=10)
        row += 1
        rule_lifts = []
        for rc in rule_cols:
            sub_rate = sub[rc].astype(float).mean() * 100
            clean_rate = clean[rc].astype(float).mean() * 100
            lift = sub_rate / max(clean_rate, 0.01)
            if sub_rate > 1:
                rule_lifts.append((rc, sub_rate, clean_rate, lift))
        rule_lifts.sort(key=lambda x: x[3], reverse=True)
        for rc, sr, cr, lf in rule_lifts[:8]:
            bg = C_RED_LIGHT if lf > 5 else C_WHITE
            write_row_styled(ws, row, [rc, f"{sr:.1f}%", f"{cr:.1f}%", f"{lf:.1f}x"], bg=bg)
            row += 1

        row += 2

    set_col_widths(ws, [(1,42),(2,18),(3,18),(4,12),(5,14),(6,14)])
    ws.sheet_view.showGridLines = False
    print("  ✓ Sheet 10_Fraud_Type_Analysis")



## 18. Sheet 11 -- Top Correlations


In [18]:
def sheet_correlations(wb, df):
    ws = wb.create_sheet("11_Top_Correlations")
    ws.freeze_panes = "A3"
    title_cell(ws, 1, 1, "Feature Correlations with AML Label")

    if COL_AML not in df.columns:
        ws.cell(row=3, column=1, value="No AML label column found").font = body_font()
        print("  ✓ Sheet 11_Top_Correlations (skipped)")
        return

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if not c.startswith("_") and c != COL_AML]
    corr = df[numeric_cols + [COL_AML]].corr()[COL_AML].drop(COL_AML).sort_values(key=abs, ascending=False)
    corr_df = corr.reset_index()
    corr_df.columns = ["feature","pearson_r"]
    corr_df["abs_r"] = corr_df["pearson_r"].abs()
    corr_df["direction"] = corr_df["pearson_r"].apply(lambda x: "Positive (↑ with AML)" if x > 0 else "Negative (↓ with AML)")
    corr_df["group"] = corr_df["feature"].map(lambda c: DATA_DICT.get(c, ("","","",""))[1])

    def signal_strength(r):
        if r > 0.5:  return "Very Strong"
        if r > 0.3:  return "Strong"
        if r > 0.15: return "Moderate"
        if r > 0.05: return "Weak"
        return "Negligible"
    corr_df["signal"] = corr_df["abs_r"].apply(signal_strength)

    apply_header(ws, 2, ["Rank","Feature","Feature Group","Pearson r","|r|","Direction","Signal Strength"], bg=C_DARK_BLUE)
    row = 3
    for rank, (_, r) in enumerate(corr_df.iterrows(), 1):
        abs_r = r["abs_r"]
        if abs_r > 0.5:   bg = "FCE4D6"
        elif abs_r > 0.3: bg = "FFF3E0"
        elif abs_r > 0.15:bg = "FFF9C4"
        else:             bg = C_WHITE
        write_row_styled(ws, row, [rank, r["feature"], r["group"], round(r["pearson_r"],4),
                         round(abs_r,4), r["direction"], r["signal"]], bg=bg)
        row += 1

    set_col_widths(ws, [(1,8),(2,42),(3,24),(4,14),(5,12),(6,28),(7,18)])
    ws.sheet_view.showGridLines = False
    print("  ✓ Sheet 11_Top_Correlations")



## 19. Create Workbook & Build All Sheets


In [19]:
wb = Workbook()
wb.remove(wb.active)
print('Workbook created.')



Workbook created.


## 20. Run All Sheet Functions


In [20]:
print(f'\nGenerating EDA Excel → {OUTPUT_PATH}\n')
sheet_overview(wb, df)
sheet_data_dict(wb, df)
sheet_formulas(wb)
sheet_desc_stats(wb, df)
sheet_missing(wb, df)
sheet_graph_eda(wb, df)
sheet_categoricals(wb, df)
sheet_rules(wb, df)
sheet_temporal(wb, df)
sheet_fraud_types(wb, df)
sheet_correlations(wb, df)

wb.save(OUTPUT_PATH)
print(f'\n✅  Saved: {OUTPUT_PATH}')
print(f'   Size: {os.path.getsize(OUTPUT_PATH)/1024:.1f} KB')




Generating EDA Excel → eda_output\AML_EDA_Summary.xlsx

  ✓ Sheet 1_Overview
  ✓ Sheet 2_Data_Dictionary
  ✓ Sheet 3_Feature_Formulas
  ✓ Sheet 4_Descriptive_Stats
  ✓ Sheet 5_Missing_Values
  ✓ Sheet 6_Graph_Features_EDA
  ✓ Sheet 7_Categorical_Distributions
  ✓ Sheet 8_Rule_Features
  ✓ Sheet 9_Temporal_Analysis
  ✓ Sheet 10_Fraud_Type_Analysis
  ✓ Sheet 11_Top_Correlations

✅  Saved: eda_output\AML_EDA_Summary.xlsx
   Size: 81.4 KB
